# NOTEBOOK RFM – Tính các chỉ số Recency, Frequency, Monetary
**Người thực hiện:** Phan Nguyễn Gia Huy  
**Học phần:** Phân tích dữ liệu lớn – Trường Đại học Quy Nhơn

## Mục tiêu
Từ bảng giao dịch đã làm sạch (`data/clean/transactions_clean.parquet`), tính 3 chỉ số RFM cho **mỗi khách hàng**:
1. Xác định ngày tham chiếu.
2. Tính Recency (R), Frequency (F), Monetary (M).
3. Kiểm tra số lượng khách hàng và giá trị RFM.
4. Lập bảng thống kê và lưu kết quả cho bước chuẩn hóa/K-Means (Chí) và vẽ biểu đồ (Đạt).

**Đầu vào:** `data/clean/transactions_clean.parquet`  
**Đầu ra:** `data/rfm/rfm_table.parquet` (cột `customer_id, R, F, M`), `reports/thong_ke_rfm.csv`

## Định nghĩa 3 chỉ số (bộ dữ liệu H&M)
| Chỉ số | Tên | Cách tính | Ý nghĩa |
|---|---|---|---|
| R | Recency | Ngày tham chiếu − ngày mua gần nhất (`max(t_dat)`), tính bằng ngày | R càng nhỏ, khách mua càng gần đây |
| F | Frequency | Số **ngày mua khác nhau** của khách (`countDistinct(t_dat)`) | F càng lớn, khách mua càng thường xuyên |
| M | Monetary | Tổng `price` của khách (`sum(price)`) | M càng lớn, khách chi càng nhiều |

**Lý do chọn cách tính:**
- Bộ H&M không có mã hóa đơn (`Invoice`) và số lượng (`Quantity`). Mỗi dòng giao dịch là một món hàng.
- Vì vậy một ngày mua được coi là một lần mua (nhiều món mua cùng ngày chỉ tính 1 lần cho F).
- `price` trong H&M là giá đã được chuẩn hóa, không có đơn vị tiền tệ thực. M chỉ dùng để so sánh tương đối giữa các khách.
- Các dòng trùng hoàn toàn được giữ lại ở bước làm sạch (cùng khách mua nhiều cái giống nhau trong ngày), nên M cộng đủ các dòng này.

In [6]:
import os
import sys
from pathlib import Path
from datetime import timedelta

from pyspark.sql import functions as F

# Tự tìm thư mục gốc dự án (nơi có src/config.py), chạy được từ bất kỳ máy nào
ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src" / "config.py").exists():
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án (thiếu src/config.py). Hãy mở notebook trong dự án.")
sys.path.append(str(ROOT / "src"))

CLEAN = ROOT / "data" / "clean"
RFM_DIR = ROOT / "data" / "rfm"
REPORTS = ROOT / "reports"
RFM_DIR.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# Khởi tạo Spark: ưu tiên spark_session.py của nhóm, nếu không có thì tạo trực tiếp
try:
    import spark_session
    if hasattr(spark_session, "get_spark"):
        spark = spark_session.get_spark("HM_RFM")
    elif hasattr(spark_session, "create_spark_session"):
        spark = spark_session.create_spark_session("HM_RFM")
    else:
        spark = spark_session.spark
except Exception:
    from pyspark.sql import SparkSession
    spark = (SparkSession.builder.appName("HM_RFM").master("local[*]")
             .config("spark.driver.memory", "8g").getOrCreate())

print("ROOT :", ROOT)
print("Spark:", spark.version)

ROOT : d:\bigdata\bán hàng\bao-cao-rfm
Spark: 4.2.0


In [7]:
# nạp dữ liệu đã làm sạch
trans_path = CLEAN / "transactions_clean.parquet"
if not trans_path.exists():
    raise FileNotFoundError(f"Chưa có {trans_path}. Hãy chạy notebook 01_kham_pha_va_lam_sach trước.")

trans = spark.read.parquet(str(trans_path)).select("customer_id", "t_dat", "price")
print(f"Số dòng giao dịch: {trans.count():,}")
trans.printSchema()
trans.show(5, truncate=False)

Số dòng giao dịch: 15,453,651
root
 |-- customer_id: long (nullable = true)
 |-- t_dat: date (nullable = true)
 |-- price: double (nullable = true)

+-----------+----------+------------------+
|customer_id|t_dat     |price             |
+-----------+----------+------------------+
|45         |2019-09-13|0.0406610169491525|
|45         |2019-09-28|0.0325254237288135|
|45         |2019-09-28|0.0216779661016949|
|45         |2019-09-28|0.0216779661016949|
|45         |2019-09-28|0.0216779661016949|
+-----------+----------+------------------+
only showing top 5 rows


In [ ]:
# ngày tham chiếu để tính RFM: ngày sau giao dịch cuối cùng 1 ngày'
max_date = trans.agg(F.max("t_dat")).collect()[0][0]
min_date = trans.agg(F.min("t_dat")).collect()[0][0]
ref_date = max_date + timedelta(days=1)

print("Ngày giao dịch đầu tiên :", min_date)
print("Ngày giao dịch cuối cùng:", max_date)
print("Ngày tham chiếu         :", ref_date)

Ngày giao dịch đầu tiên : 2019-09-07
Ngày giao dịch cuối cùng: 2020-09-22
Ngày tham chiếu         : 2020-09-23


In [9]:
# tính RFM
rfm = (
    trans.groupBy("customer_id")
    .agg(
        F.datediff(F.lit(ref_date), F.max("t_dat")).alias("R"),
        F.countDistinct("t_dat").alias("F"),
        F.round(F.sum("price"), 4).alias("M"),
    )
    .cache()
)
n_rfm = rfm.count()
print(f"Số khách trong bảng RFM: {n_rfm:,}")
rfm.show(5, truncate=False)

Số khách trong bảng RFM: 1,007,704
+-----------+---+---+------+
|customer_id|R  |F  |M     |
+-----------+---+---+------+
|406        |4  |6  |0.5201|
|670        |83 |5  |0.4759|
|997        |72 |13 |0.8005|
|1025       |214|2  |0.0406|
|1469       |14 |15 |4.8184|
+-----------+---+---+------+
only showing top 5 rows


In [10]:
n_trans_cust = trans.select("customer_id").distinct().count()
cust_path = CLEAN / "customer_clean.parquet"
n_cust_all = spark.read.parquet(str(cust_path)).count() if cust_path.exists() else None

print(f"Khách trong RFM        : {n_rfm:,}")
print(f"Khách có giao dịch     : {n_trans_cust:,}")
if n_cust_all:
    print(f"Khách trong bảng customer: {n_cust_all:,} | không có giao dịch: {n_cust_all - n_rfm:,}")
assert n_rfm == n_trans_cust, "Số khách RFM khác số khách có giao dịch!"

print("customer_id trùng:", n_rfm - rfm.select("customer_id").distinct().count())
rfm.select([F.sum(F.col(c).isNull().cast("int")).alias(c + "_null") for c in ["R", "F", "M"]]).show()
print("R < 1:", rfm.filter("R < 1").count(),
      "| F < 1:", rfm.filter("F < 1").count(),
      "| M <= 0:", rfm.filter("M <= 0").count())

sum_rfm = rfm.agg(F.sum("M")).collect()[0][0]
sum_trans = trans.agg(F.sum("price")).collect()[0][0]
print(f"Tổng M (RFM): {sum_rfm:,.4f} | Tổng price (sạch): {sum_trans:,.4f} | Chênh: {abs(sum_rfm - sum_trans):,.6f}")

Khách trong RFM        : 1,007,704
Khách có giao dịch     : 1,007,704
Khách trong bảng customer: 1,371,980 | không có giao dịch: 364,276
customer_id trùng: 0
+------+------+------+
|R_null|F_null|M_null|
+------+------+------+
|     0|     0|     0|
+------+------+------+

R < 1: 0 | F < 1: 0 | M <= 0: 0
Tổng M (RFM): 438,629.2827 | Tổng price (sạch): 438,629.4301 | Chênh: 0.147417


In [11]:
# thống kê mô tả
stats = rfm.select("R", "F", "M").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
stats.show(truncate=False)

n_f1 = rfm.filter("F = 1").count()
print(f"Khách chỉ mua 1 ngày (F=1): {n_f1:,} ({n_f1 / n_rfm * 100:.2f}%)")

for c in ["R", "F", "M"]:
    mean_v = rfm.agg(F.mean(c)).collect()[0][0]
    med_v = rfm.approxQuantile(c, [0.5], 0.01)[0]
    print(f"{c}: mean = {mean_v:,.3f} | median = {med_v:,.3f} | mean/median = {mean_v / med_v:,.2f}")

rfm.orderBy(F.desc("M")).show(10, truncate=False)

+-------+------------------+-----------------+------------------+
|summary|R                 |F                |M                 |
+-------+------------------+-----------------+------------------+
|count  |1007704           |1007704          |1007704           |
|mean   |120.85609861626033|4.510442550590253|0.4352759170351705|
|stddev |105.80636594002124|5.633151116498266|0.7028149836662015|
|min    |1                 |1                |8.0E-4            |
|25%    |32                |1                |0.0847            |
|50%    |88                |2                |0.2049            |
|75%    |193               |6                |0.4986            |
|max    |382               |233              |34.9546           |
+-------+------------------+-----------------+------------------+

Khách chỉ mua 1 ngày (F=1): 343,479 (34.09%)
R: mean = 120.856 | median = 87.000 | mean/median = 1.39
F: mean = 4.510 | median = 2.000 | mean/median = 2.26
M: mean = 0.435 | median = 0.203 | mean/median = 2.

In [ ]:
# lưu kết quả
rfm.write.mode("overwrite").parquet(str(RFM_DIR / "rfm_table.parquet"))
stats.toPandas().to_csv(REPORTS / "thong_ke_rfm.csv", index=False, encoding="utf-8-sig")

check = spark.read.parquet(str(RFM_DIR / "rfm_table.parquet"))
print("Đã lưu:", f"{check.count():,}", "dòng | cột:", check.columns)

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Đã lưu: 1,007,704 dòng | cột: ['customer_id', 'R', 'F', 'M']
